# Notebook 7: Model Evaluation
## Comprehensive Evaluation — ROC, Confusion Matrix, Bootstrap CI, Stability

**Metrics**: Accuracy, Precision, Recall, F1-Score, AUC-ROC, Confusion Matrix  
**Advanced**: Bootstrap confidence intervals, Stability analysis

In [ ]:
# ============================================================================
# Cell 1: Setup
# ============================================================================
import os, sys, time
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc, 
    precision_recall_curve
)

sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================================
# Cell 2: Spark Session
# ============================================================================
spark = (
    SparkSession.builder
    .appName('RedditVirality_Evaluation')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

In [ ]:
# ============================================================================
# Cell 3: Load Predictions from All Models
# ============================================================================
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

model_names = ['lr', 'rf', 'dt', 'gbt']
model_labels = ['Logistic Regression', 'Random Forest', 'Decision Tree', 'Gradient Boosted Trees']
predictions = {}

for name, label in zip(model_names, model_labels):
    path = os.path.join(PROCESSED_DIR, f'predictions_{name}.parquet')
    if os.path.exists(path):
        preds = spark.read.parquet(path)
        predictions[label] = preds
        print(f'Loaded {label}: {preds.count():,} predictions')
    else:
        print(f'WARNING: {path} not found. Run Notebook 4 first.')

## 1. Classification Metrics Table

In [ ]:
# ============================================================================
# Cell 4: Compute All Metrics
# ============================================================================
binary_eval = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
acc_eval = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy')
prec_eval = MulticlassClassificationEvaluator(labelCol='label', metricName='weightedPrecision')
rec_eval = MulticlassClassificationEvaluator(labelCol='label', metricName='weightedRecall')
f1_eval = MulticlassClassificationEvaluator(labelCol='label', metricName='f1')

results = []
for name, preds in predictions.items():
    metrics = {
        'Model': name,
        'Accuracy': round(acc_eval.evaluate(preds), 4),
        'Precision': round(prec_eval.evaluate(preds), 4),
        'Recall': round(rec_eval.evaluate(preds), 4),
        'F1-Score': round(f1_eval.evaluate(preds), 4),
        'AUC-ROC': round(binary_eval.evaluate(preds), 4)
    }
    results.append(metrics)

results_df = pd.DataFrame(results)
print('=' * 80)
print('CLASSIFICATION METRICS COMPARISON')
print('=' * 80)
print(results_df.to_string(index=False))

## 2. Confusion Matrices

In [ ]:
# ============================================================================
# Cell 5: Confusion Matrices for All Models
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, (name, preds) in enumerate(predictions.items()):
    # Collect predictions to local
    local_preds = preds.select('label', 'prediction').toPandas()
    cm = confusion_matrix(local_preds['label'], local_preds['prediction'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Non-Viral', 'Viral'],
                yticklabels=['Non-Viral', 'Viral'])
    axes[i].set_title(f'{name}', fontsize=13)
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — All Models', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. ROC Curves

In [ ]:
# ============================================================================
# Cell 6: ROC Curves for All Models
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for i, (name, preds) in enumerate(predictions.items()):
    # Extract probability of positive class
    from pyspark.ml.functions import vector_to_array
    
    local = preds.select(
        'label',
        vector_to_array('probability').alias('prob_array')
    ).withColumn('prob_viral', F.element_at('prob_array', 2)) \
     .select('label', 'prob_viral').toPandas()
    
    fpr, tpr, _ = roc_curve(local['label'], local['prob_viral'])
    roc_auc = auc(fpr, tpr)
    
    ax.plot(fpr, tpr, color=colors[i], lw=2,
            label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — All Models', fontsize=16)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Bootstrap Confidence Intervals

Compute 95% confidence intervals for key metrics using bootstrap resampling.  
This demonstrates the **stability and reliability** of our model estimates.

In [ ]:
# ============================================================================
# Cell 7: Bootstrap Confidence Intervals
# ============================================================================
from sklearn.metrics import accuracy_score, f1_score as sk_f1_score
from sklearn.utils import resample

N_BOOTSTRAP = 100  # Number of bootstrap iterations

def bootstrap_ci(y_true, y_pred, metric_fn, n_iterations=100, ci=0.95):
    """Compute bootstrap confidence interval for a metric."""
    scores = []
    for _ in range(n_iterations):
        indices = resample(range(len(y_true)), n_samples=len(y_true), random_state=None)
        y_t = y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices]
        y_p = y_pred.iloc[indices] if hasattr(y_pred, 'iloc') else y_pred[indices]
        scores.append(metric_fn(y_t, y_p))
    
    alpha = (1 - ci) / 2
    lower = np.percentile(scores, alpha * 100)
    upper = np.percentile(scores, (1 - alpha) * 100)
    mean = np.mean(scores)
    return mean, lower, upper

print('BOOTSTRAP CONFIDENCE INTERVALS (95%)')
print('=' * 80)

ci_results = []

for name, preds in predictions.items():
    local = preds.select('label', 'prediction').toPandas()
    y_true = local['label'].values
    y_pred = local['prediction'].values
    
    acc_mean, acc_lo, acc_hi = bootstrap_ci(y_true, y_pred, accuracy_score, N_BOOTSTRAP)
    f1_mean, f1_lo, f1_hi = bootstrap_ci(
        y_true, y_pred, lambda y, p: sk_f1_score(y, p, average='weighted'), N_BOOTSTRAP
    )
    
    ci_results.append({
        'Model': name,
        'Accuracy (Mean)': round(acc_mean, 4),
        'Accuracy 95% CI': f'[{acc_lo:.4f}, {acc_hi:.4f}]',
        'F1 (Mean)': round(f1_mean, 4),
        'F1 95% CI': f'[{f1_lo:.4f}, {f1_hi:.4f}]'
    })
    print(f'{name}:')
    print(f'  Accuracy: {acc_mean:.4f} [{acc_lo:.4f}, {acc_hi:.4f}]')
    print(f'  F1-Score: {f1_mean:.4f} [{f1_lo:.4f}, {f1_hi:.4f}]')

ci_df = pd.DataFrame(ci_results)
print('\n' + ci_df.to_string(index=False))

## 5. Stability Analysis

Test how model performance changes with different random data subsets.  
A stable model shows low variance across different samples.

In [ ]:
# ============================================================================
# Cell 8: Stability Analysis (Data Perturbation)
# ============================================================================
print('STABILITY ANALYSIS — Data Perturbation Test')
print('=' * 60)
print('Testing: Does small data change → big model change?')

# For each model, subsample test data 10 times and measure metric variation
N_STABILITY_RUNS = 10
SUBSAMPLE_FRACTION = 0.8

stability_data = []

for name, preds in predictions.items():
    local = preds.select('label', 'prediction').toPandas()
    accuracies = []
    f1_scores = []
    
    for i in range(N_STABILITY_RUNS):
        sample = local.sample(frac=SUBSAMPLE_FRACTION, random_state=i)
        acc = accuracy_score(sample['label'], sample['prediction'])
        f1 = sk_f1_score(sample['label'], sample['prediction'], average='weighted')
        accuracies.append(acc)
        f1_scores.append(f1)
        stability_data.append({
            'Model': name, 'Run': i+1, 'Accuracy': acc, 'F1-Score': f1
        })
    
    acc_std = np.std(accuracies)
    f1_std = np.std(f1_scores)
    print(f'{name}:')
    print(f'  Accuracy: mean={np.mean(accuracies):.4f}, std={acc_std:.4f}')
    print(f'  F1:       mean={np.mean(f1_scores):.4f}, std={f1_std:.4f}')
    print(f'  Stability: {"HIGH" if f1_std < 0.005 else "MODERATE" if f1_std < 0.01 else "LOW"}')

stability_df = pd.DataFrame(stability_data)

In [ ]:
# ============================================================================
# Cell 9: Stability Visualization
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=stability_df, x='Model', y='Accuracy', ax=axes[0], palette='viridis')
axes[0].set_title('Accuracy Stability Across 10 Subsamples', fontsize=13)
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=stability_df, x='Model', y='F1-Score', ax=axes[1], palette='viridis')
axes[1].set_title('F1-Score Stability Across 10 Subsamples', fontsize=13)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'stability_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# Cell 10: Export All Evaluation Data for Tableau
# ============================================================================
TABLEAU_DIR = os.path.join(PROJECT_ROOT, 'tableau', 'data')
os.makedirs(TABLEAU_DIR, exist_ok=True)

# Metrics table
results_df.to_csv(os.path.join(TABLEAU_DIR, 'evaluation_metrics.csv'), index=False)

# Confidence intervals
ci_df.to_csv(os.path.join(TABLEAU_DIR, 'confidence_intervals.csv'), index=False)

# Stability data
stability_df.to_csv(os.path.join(TABLEAU_DIR, 'stability_analysis.csv'), index=False)

# Confusion matrix data for each model
for name, preds in predictions.items():
    local = preds.select('label', 'prediction').toPandas()
    cm = confusion_matrix(local['label'], local['prediction'])
    cm_df = pd.DataFrame(cm, 
        index=['Actual_NonViral', 'Actual_Viral'],
        columns=['Pred_NonViral', 'Pred_Viral']
    )
    safe_name = name.replace(' ', '_').lower()
    cm_df.to_csv(os.path.join(TABLEAU_DIR, f'confusion_matrix_{safe_name}.csv'))

print('All evaluation data exported to tableau/data/')
print(f'Files: {os.listdir(TABLEAU_DIR)}')

spark.stop()
print('Spark session stopped.')